# Phase 5-6: Knowledge Graph Construction & Embedding Initialization

This notebook:
- **Phase 5**: Builds Knowledge Graph with temporal decay weights
- **Phase 6**: Initializes KG nodes with sentence embedding features

**Run Time**: ~15 mins
**Input**: Outputs from Phase 2-3
**Output**: KG structure + node embeddings

In [1]:
# Cell 1: Install dependencies
!pip install -q networkx pandas numpy pickle scikit-learn tqdm -q

ERROR: Could not find a version that satisfies the requirement pickle (from versions: none)
ERROR: No matching distribution found for pickle


In [2]:
# Cell 2: Imports
import os
import pickle
import pandas as pd
import numpy as np
import networkx as nx
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Environment detection - local vs Kaggle
BASE_DIR = os.getcwd()
if os.path.exists('/kaggle/input/datasets/chandrimanandi/phase-2-3-results/train.csv'):
    # Real Kaggle environment
    INPUT_DIR = '/kaggle/input/datasets/chandrimanandi/phase-2-3-results'
    OUTPUT_DIR = '/kaggle/working'
else:
    # Local environment
    INPUT_DIR = os.path.join(BASE_DIR, 'output')
    OUTPUT_DIR = os.path.join(BASE_DIR, 'output', 'phase-5-6-results')

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"INPUT_DIR: {INPUT_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

SEED = 42
np.random.seed(SEED)
print("Setup complete")

INPUT_DIR: /kaggle/input/datasets/chandrimanandi/phase-2-3-results
OUTPUT_DIR: /kaggle/working
Setup complete


In [ ]:
# Cell 3: Load data
print("Loading preprocessed data...")

train_df = pd.read_csv(f"{INPUT_DIR}/train.csv")

with open(f"{INPUT_DIR}/id_maps.pkl", 'rb') as f:
    id_maps = pickle.load(f)

# Load embeddings if available (BERT-only system: no Sentence Transformers)
try:
    with open(f"{INPUT_DIR}/embeddings.pkl", 'rb') as f:
        embeddings_data = pickle.load(f)
    # Try to get item_embeddings, fall back to random init if unavailable
    if 'item_embeddings' in embeddings_data:
        item_embeddings = embeddings_data['item_embeddings']
        embedding_dim = embeddings_data.get('embedding_dim', 64)
        embeddings_available = True
        print("✓ Using saved item embeddings for KG initialization")
    else:
        print("⚠ No item embeddings found - using random initialization")
        embedding_dim = 64
        item_embeddings = {}
        embeddings_available = False
except FileNotFoundError:
    print("⚠ embeddings.pkl not found - using random initialization")
    embedding_dim = 64
    item_embeddings = {}
    embeddings_available = False

n_users = id_maps['n_users']
n_items = id_maps['n_items']

print(f"Users: {n_users}, Items: {n_items}")
print(f"Embedding dimension: {embedding_dim}")
print(f"Embeddings available: {embeddings_available}")

Loading preprocessed data...
Users: 5541, Items: 3568
Item embeddings: 3568
Embedding dimension: 384


## PHASE 5: Knowledge Graph Construction

### KG Structure Built:
1. **User Nodes**: One per user (n_users nodes)
2. **Item Nodes**: One per item (n_items nodes) 
3. **Attribute Nodes**: Categories, Brands, Genres, Artists
4. **Edge Types**:
   - User → Item (interaction edges, weighted by temporal decay)
   - Item → Item (through shared metadata)
   - Item → Category (weight=1.0)
   - Item → Brand (weight=1.0)
   - Item → Genre (weight=1.0)
   - Item → Artist (weight=1.0)

### Node Initialization (BERT-Only System):
- **Item nodes**: Initialized with saved embeddings if available (random fallback if missing)
- **User nodes**: Zero embeddings (learned via random walks in node2vec)
- **Attribute nodes**: Zero embeddings (learned via node2vec propagation)
- No Sentence Transformers used - using BERT-only system

In [4]:
# Cell 4: Build KG with temporal decay weights
print("\n" + "="*60)
print("PHASE 5: BUILDING KNOWLEDGE GRAPH")
print("="*60)

LAMBDA = 1e-7  # Temporal decay rate

G = nx.DiGraph()

# Add nodes
print("Adding nodes...")
for uid in tqdm(range(n_users), desc="User nodes"):
    G.add_node(f'user_{uid}', node_type='user')

for iid in tqdm(range(n_items), desc="Item nodes"):
    # Initialize with sentence embedding
    emb = item_embeddings.get(iid, np.zeros(embedding_dim))
    G.add_node(f'item_{iid}', node_type='item', features=emb)

print(f"Nodes added: {G.number_of_nodes():,}")


PHASE 5: BUILDING KNOWLEDGE GRAPH
Adding nodes...


Item nodes: 100%|██████████| 3568/3568 [00:00<00:00, 232654.63it/s]

Nodes added: 9,109


In [5]:
# Cell 5: Add edges with temporal decay weights
print("\nAdding edges with temporal decay...")

t_max = train_df['timestamp'].max() if 'timestamp' in train_df.columns else 0
print(f"Max timestamp: {t_max}")

edge_count = 0
weights = []

for _, row in tqdm(train_df.iterrows(), total=len(train_df), desc="Adding edges"):
    u_id = int(row['user_idx'])
    i_id = int(row['item_idx'])
    
    u_node = f'user_{u_id}'
    i_node = f'item_{i_id}'
    
    if 'timestamp' in train_df.columns:
        timestamp = row['timestamp']
        weight = np.exp(-LAMBDA * (t_max - timestamp))
    else:
        weight = 1.0
    
    weights.append(weight)
    
    # Bidirectional edges
    G.add_edge(u_node, i_node, weight=weight)
    G.add_edge(i_node, u_node, weight=weight)
    
    edge_count += 2

print(f"Edges added: {edge_count:,}")
print(f"Graph stats: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges")
print(f"Edge weights: min={min(weights):.6f}, max={max(weights):.6f}, mean={np.mean(weights):.6f}")


Adding edges with temporal decay...
Max timestamp: 1405900800


Adding edges: 100%|██████████| 53624/53624 [00:02<00:00, 18533.40it/s]

Edges added: 107,248
Graph stats: 9,109 nodes, 107,248 edges
Edge weights: min=0.000000, max=1.000000, mean=0.010649


In [6]:
# Cell 5b: Load metadata mappings
print("\nLoading metadata for KG enrichment...")

try:
    with open(f"{INPUT_DIR}/metadata_mappings.pkl", 'rb') as f:
        metadata_mappings = pickle.load(f)
    print("✓ Metadata mappings loaded")
    print(f"  Categories: {len(metadata_mappings['categories'])}")
    print(f"  Brands: {len(metadata_mappings['brands'])}")
    print(f"  Genres: {len(metadata_mappings['genres'])}")
    print(f"  Artists: {len(metadata_mappings['artists'])}")
except FileNotFoundError:
    print("⚠ Metadata mappings not found - creating empty mappings")
    metadata_mappings = {
        'item_to_category': {},
        'item_to_brand': {},
        'item_to_genre': {},
        'item_to_artist': {},
        'categories': [],
        'brands': [],
        'genres': [],
        'artists': [],
    }



Loading metadata for KG enrichment...
✓ Metadata mappings loaded
  Categories: 1
  Brands: 1
  Genres: 1
  Artists: 1


In [7]:
# Cell 6: Verify KG connectivity
print("\nKG connectivity analysis...")

# Check connected components
n_components = nx.number_strongly_connected_components(G)
n_weakly_components = nx.number_weakly_connected_components(G)
print(f"Strongly connected components: {n_components}")
print(f"Weakly connected components: {n_weakly_components}")

# Degree statistics
degrees = [d for _, d in G.degree()]
print(f"Degree stats: min={min(degrees)}, max={max(degrees)}, mean={np.mean(degrees):.2f}")

# Check disconnected nodes
isolated_nodes = list(nx.isolates(G))
print(f"Isolated nodes: {len(isolated_nodes)}")

if isolated_nodes:
    print(f"  (Note: Some nodes may have no interactions in this split)")


KG connectivity analysis...
Strongly connected components: 4
Weakly connected components: 4
Degree stats: min=0, max=1152, mean=23.55
Isolated nodes: 3
  (Note: Some nodes may have no interactions in this split)


In [8]:
# Cell 6b: Add metadata attribute nodes and edges
print("\n" + "="*60)
print("ADDING METADATA NODES & EDGES")
print("="*60)

# Add category nodes and edges
print("\n1. Adding category nodes and edges...")
category_count = 0
for item_id in tqdm(range(n_items), desc="Item-Category edges", leave=False):
    category = metadata_mappings['item_to_category'].get(item_id, "Unknown")
    cat_node = f'category_{category}'
    
    # Add category node if not exists
    if cat_node not in G:
        G.add_node(cat_node, node_type='category', features=np.zeros(embedding_dim))
        category_count += 1
    
    # Add bidirectional edges with weight 1.0 (metadata edge)
    G.add_edge(f'item_{item_id}', cat_node, weight=1.0)
    G.add_edge(cat_node, f'item_{item_id}', weight=1.0)

print(f"  Added: {category_count} category nodes")

# Add brand nodes and edges
print("\n2. Adding brand nodes and edges...")
brand_count = 0
for item_id in tqdm(range(n_items), desc="Item-Brand edges", leave=False):
    brand = metadata_mappings['item_to_brand'].get(item_id, "Unknown")
    brand_node = f'brand_{brand}'
    
    # Add brand node if not exists
    if brand_node not in G:
        G.add_node(brand_node, node_type='brand', features=np.zeros(embedding_dim))
        brand_count += 1
    
    # Add bidirectional edges with weight 1.0 (metadata edge)
    G.add_edge(f'item_{item_id}', brand_node, weight=1.0)
    G.add_edge(brand_node, f'item_{item_id}', weight=1.0)

print(f"  Added: {brand_count} brand nodes")

# Add artist nodes and edges
print("\n3. Adding artist nodes and edges...")
artist_count = 0
for item_id in tqdm(range(n_items), desc="Item-Artist edges", leave=False):
    artist = metadata_mappings['item_to_artist'].get(item_id, "Unknown")
    artist_node = f'artist_{artist}'
    
    # Add artist node if not exists
    if artist_node not in G:
        G.add_node(artist_node, node_type='artist', features=np.zeros(embedding_dim))
        artist_count += 1
    
    # Add bidirectional edges with weight 1.0 (metadata edge)
    G.add_edge(f'item_{item_id}', artist_node, weight=1.0)
    G.add_edge(artist_node, f'item_{item_id}', weight=1.0)

print(f"  Added: {artist_count} artist nodes")

print(f"\n✓ Metadata integration complete!")
print(f"  Total categories: {len(metadata_mappings['categories'])}")
print(f"  Total brands: {len(metadata_mappings['brands'])}")
print(f"  Total artists: {len(metadata_mappings['artists'])}")
print(f"\n  Total KG nodes: {G.number_of_nodes():,}")
print(f"  Total KG edges: {G.number_of_edges():,}")

# Node type distribution
from collections import Counter
node_types = Counter([G.nodes[n].get('node_type', 'unknown') for n in G.nodes()])
print(f"\nNode type distribution:")
for node_type, count in sorted(node_types.items()):
    print(f"  {node_type}: {count:,}")



ADDING METADATA NODES & EDGES

1. Adding category nodes and edges...


  Added: 1 category nodes

2. Adding brand nodes and edges...


  Added: 1 brand nodes

3. Adding artist nodes and edges...


  Added: 1 artist nodes

✓ Metadata integration complete!
  Total categories: 1
  Total brands: 1
  Total artists: 1

  Total KG nodes: 9,112
  Total KG edges: 128,656

Node type distribution:
  artist: 1
  brand: 1
  category: 1
  item: 3,568
  user: 5,541


## PHASE 6: KG Initialization with Embeddings

In [ ]:
# Cell 7: Initialize item nodes with embeddings
print("\n" + "="*60)
print("PHASE 6: KG INITIALIZATION WITH EMBEDDINGS")
print("="*60)

print("\nInitializing KG nodes...")

node_features = {}

# Item nodes get saved embeddings if available, else random init
for item_id in range(n_items):
    node_name = f'item_{item_id}'
    if node_name in G:
        if embeddings_available and item_id in item_embeddings:
            emb = item_embeddings.get(item_id)
        else:
            # Random initialization for missing embeddings
            emb = np.random.randn(embedding_dim) * 0.01
        G.nodes[node_name]['features'] = emb
        node_features[node_name] = emb

# User nodes get zero embeddings (learned via random walks)
for user_id in range(n_users):
    node_name = f'user_{user_id}'
    if node_name in G:
        G.nodes[node_name]['features'] = np.zeros(embedding_dim)
        node_features[node_name] = np.zeros(embedding_dim)

print(f"Node features initialized for: {len(node_features)}")
print(f"  Using {"actual embeddings" if embeddings_available else "random initialization"} for {n_items} items")

# Verify all nodes have features
nodes_with_features = sum(1 for n in G.nodes() if 'features' in G.nodes[n])
print(f"Nodes with features: {nodes_with_features} / {G.number_of_nodes()}")


PHASE 6: KG INITIALIZATION WITH EMBEDDINGS

Initializing item nodes with sentence embeddings...
Node features initialized for: 9109
Nodes with features: 9112 / 9112


### Review-to-KG Embedding Pipeline (with Real Amazon Metadata):

**Step 1 - Load Amazon Product Metadata**:
```
Product ASIN → title, brand, categories (from meta_Digital_Music.json)
Extract: Category (first in list), Brand, Artist (from title)
```

**Step 2 - Review Text Encoding (Phase 3)**:
```
Review text → Sentence Transformer (all-MiniLM-L6-v2) → 384-dim embedding
```

**Step 3 - Item Aggregation (Phase 3)**:
```
Multiple reviews per item → Average embeddings → Item representation
Reviews semantics captured in single vector
```

**Step 4 - KG Initialization (Phase 5)**:
```
Item nodes ← Item embeddings (semantic review information)
Attribute nodes:
  - Category nodes (from Amazon metadata)
  - Brand nodes (from Amazon metadata)
  - Artist nodes (extracted from product title)
User nodes ← Zero vectors (will be learned via random walks)
```

**Step 5 - KG Structure**:
- **Nodes**: Items + Users + Categories + Brands + Artists
- **Edges**: 
  - User-Item (temporal weighted)
  - Item-Category (weight=1.0)
  - Item-Brand (weight=1.0)
  - Item-Artist (weight=1.0)

**Step 6 - node2vec Refinement (Phase 7)**:
```
Random walks through KG
↓
Capture structure: 
  - Items in same category → nearby in walks
  - Items by same brand → nearby in walks
  - Items by same artist → nearby in walks
↓
Skip-gram learning: nodes in walks become similar embeddings
↓
Final: Item embeddings = semantic info (reviews) + structural info (metadata)
```

**Key Innovation - NLP ↔ KG Coupling**:
1. Reviews encode semantic product information
2. Sentence Transformer captures meaning
3. KG nodes start with this semantic knowledge
4. node2vec refines based on REAL product metadata structure
5. Final embeddings = review semantics + Amazon metadata relationships

In [10]:
# Cell 8: Verify embedding quality
print("\nEmbedding quality checks...")

# Check for NaN embeddings
nan_count = 0
for node in G.nodes():
    if 'features' in G.nodes[node]:
        if np.isnan(G.nodes[node]['features']).any():
            nan_count += 1

print(f"NaN embeddings: {nan_count}")

# Embedding statistics
all_embs = np.array([G.nodes[n]['features'] for n in G.nodes() if 'features' in G.nodes[n]])
print(f"Embedding stats:")
print(f"  Min value: {all_embs.min():.6f}")
print(f"  Max value: {all_embs.max():.6f}")
print(f"  Mean value: {all_embs.mean():.6f}")
print(f"  Std dev: {all_embs.std():.6f}")


Embedding quality checks...
NaN embeddings: 0
Embedding stats:
  Min value: -0.207951
  Max value: 0.196915
  Mean value: -0.000140
  Std dev: 0.024627


In [11]:
# Cell 9: Build node embedding matrix for fast access
print("\nBuilding node embedding matrices...")

# Item embedding matrix
item_emb_matrix = np.zeros((n_items, embedding_dim))
for item_id in range(n_items):
    item_emb_matrix[item_id] = item_embeddings.get(item_id, np.zeros(embedding_dim))

# Normalize for cosine similarity
from sklearn.preprocessing import normalize
item_emb_normalized = normalize(item_emb_matrix, norm='l2')

print(f"Item embedding matrix shape: {item_emb_matrix.shape}")
print(f"Normalized: {item_emb_normalized.shape}")


Building node embedding matrices...
Item embedding matrix shape: (3568, 384)
Normalized: (3568, 384)


### Edge Weight Scheme:

**User-Item Edges** (Interaction-based):
- Formula: `w = exp(-λ × (t_max - timestamp))`
- λ = 1e-7 (temporal decay parameter)
- Recent interactions: weight → 1.0
- Older interactions: weight → 0.0 (exponential decay)
- Rationale: Recent behavior more important for sequential patterns

**Metadata Edges** (Item-Category, Item-Brand, Item-Genre, Item-Artist):
- Weight: 1.0 (uniform)
- Rationale: Structural relationships don't decay
- Same metadata → Equal connection strength in random walks

**Impact on Random Walks**:
- Higher weight = more likely to traverse that edge
- node2vec respects edge weights during walk sampling
- Result: Weighted by interaction recency + metadata connections

In [12]:
# Cell 10: Save all outputs
print("\n" + "="*60)
print("SAVING OUTPUTS")
print("="*60)

# Save KG
print("Saving KG...")
with open(f"{OUTPUT_DIR}/knowledge_graph.pkl", 'wb') as f:
    pickle.dump(G, f)
print("✓ Knowledge graph saved")

# Save node features separately (for faster access)
print("Saving node features...")
node_features_dict = {}
for node in G.nodes():
    if 'features' in G.nodes[node]:
        node_features_dict[node] = G.nodes[node]['features']

with open(f"{OUTPUT_DIR}/node_features.pkl", 'wb') as f:
    pickle.dump(node_features_dict, f)
print("✓ Node features saved")

# Save embeddings
print("Saving embedding matrices...")
with open(f"{OUTPUT_DIR}/kg_embeddings.pkl", 'wb') as f:
    pickle.dump({
        'item_emb_matrix': item_emb_matrix,
        'item_emb_normalized': item_emb_normalized,
        'embedding_dim': embedding_dim,
    }, f)
print("✓ KG embeddings saved")

# Save KG statistics
import json
kg_stats = {
    'n_nodes': G.number_of_nodes(),
    'n_edges': G.number_of_edges(),
    'n_users': n_users,
    'n_items': n_items,
    'n_item_nodes_with_features': sum(1 for i in range(n_items) if f'item_{i}' in node_features_dict),
    'embedding_dim': embedding_dim,
    'temporal_lambda': LAMBDA,
}

with open(f"{OUTPUT_DIR}/kg_stats.json", 'w') as f:
    json.dump(kg_stats, f, indent=2)
print("✓ KG statistics saved")

print(f"\n{'='*60}")
print(f"PHASE 5-6 COMPLETE")
print(f"{'='*60}")
print(f"KG nodes: {G.number_of_nodes():,}")
print(f"KG edges: {G.number_of_edges():,}")
print(f"Item nodes initialized: {kg_stats['n_item_nodes_with_features']}")
print(f"Embedding dimension: {embedding_dim}")
print(f"\nAll files saved to: {OUTPUT_DIR}")


SAVING OUTPUTS
Saving KG...
✓ Knowledge graph saved
Saving node features...
✓ Node features saved
Saving embedding matrices...
✓ KG embeddings saved
✓ KG statistics saved

PHASE 5-6 COMPLETE
KG nodes: 9,112
KG edges: 128,656
Item nodes initialized: 3568
Embedding dimension: 384

All files saved to: /kaggle/working
